# Pixels In, Points Out: Training a Real Pong Agent

In the CartPole notebook you built the *shape* of every reinforcement learning problem: **observe the state -> choose an action -> get a reward -> repeat.** CartPole's state was 4 clean numbers, so a hand-written one-line rule already beat random guessing.

Pong doesn't give you 4 clean numbers. It gives you a 210x160 color screen, 60 times a second, and no rule you could write by hand will tell you "the ball is at position X, moving in direction Y." You have to **learn** that from pixels - which is exactly why every notebook before this one has been teaching you CNNs.

**This notebook combines both halves of the leerlijn for real:**
- The **CNN** (from the digit/MNIST/CIFAR-10 notebooks) turns raw pixels into useful features.
- The **RL loop** (from CartPole) turns those features into a decision, using **Deep Q-Learning (DQN)** - the same core idea (the Bellman equation, learning Q-values) from your course book, just with a neural network estimating the Q-values instead of a table.

This is the real, unscaffolded version of the DLMAIRIL01 Task 2 assignment (Playing Pong). Treat this notebook as your working implementation - the write-up (architecture choices, hyperparameters, results, honest discussion of limitations) is the part you still do yourself for the actual paper.

In [ ]:
# Colab setup. Locally-tested equivalent of this exact pipeline (env, wrappers,
# network, replay buffer, training step) already ran end-to-end during
# development - this cell just gets the same packages into this notebook.
!pip install -q "ale-py>=0.9" "gymnasium[atari]" autorom
!AutoROM --accept-license > /dev/null 2>&1

import random
import time
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import ale_py

gym.register_envs(ale_py)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, then re-run this cell.")

## Step 1 - From 4 numbers to a screen full of pixels

> 🏷️ **B1-K1-W2** · Maakt een technisch ontwerp voor software — kwalificatiedossier Software development, kerntaak B1-K1 (de ontwerpkeuze voor input-representatie)

Three standard preprocessing choices, all handled by Gymnasium's built-in Atari wrappers so we don't hand-roll something fragile:

- **`AtariPreprocessing`**: skips 4 frames per action (the game barely changes frame to frame, so this is mostly wasted computation), converts to grayscale (color doesn't help tell the ball from the paddle), and resizes to 84x84 (way cheaper for a CNN than 210x160).
- **`FrameStackObservation`**: stacks the last 4 frames together. A *single* frame can't show you which way the ball is moving - 4 frames in a row can, the same way a flipbook shows motion from still pictures.

The result: instead of CartPole's `(4,)` array of numbers, our "state" is now a `(4, 84, 84)` stack of grayscale frames - and the network has to figure out ball position, paddle position, and direction of motion purely from that, the same way the CNN notebooks learned to recognize digits purely from pixel brightness.

In [ ]:
def make_env(render_mode=None):
    env = gym.make("ALE/Pong-v5", frameskip=1, render_mode=render_mode)
    env = AtariPreprocessing(env, frame_skip=4, screen_size=84, grayscale_obs=True, scale_obs=False)
    env = FrameStackObservation(env, stack_size=4)
    return env

env = make_env()
n_actions = env.action_space.n
state, info = env.reset()
print("State shape:", state.shape, state.dtype)
print("Number of actions:", n_actions, "->", env.unwrapped.get_action_meanings())

## Step 2 - Seeing what the network sees

Before trusting a network to learn from this, let's actually look at it. Play a handful of random steps and plot all 4 stacked frames - you should be able to see the paddles and ball nudge slightly between frames.

In [ ]:
state, info = env.reset()
for _ in range(20):
    state, reward, terminated, truncated, info = env.step(env.action_space.sample())

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for i, ax in enumerate(axes):
    ax.imshow(state[i], cmap="gray")
    ax.set_title(f"frame t-{3-i}")
    ax.axis("off")
plt.suptitle("The 4 stacked frames the network actually sees as one 'state'")
plt.tight_layout()
plt.show()

## Step 3 - The Q-network

> 🏷️ **B1-K1-W3** · Realiseert (onderdelen van) software — kwalificatiedossier Software development, kerntaak B1-K1

Same conv -> pool -> ... shape you already know, with two differences from the digit-classifying CNNs:

1. There's no pooling. Atari DQN traditionally uses **strided convolutions** instead (the `stride=4`, `stride=2` below) to shrink the image, since exact spatial position matters more for a game than for digit classification.
2. The output isn't "which digit is this" - it's **one number per action**, the network's current estimate of "how much total future reward do I get if I take this action right now." That's the Q-value. Whichever action has the highest Q-value is the network's current best guess at what to do.

This is the exact architecture from the 2015 DeepMind Atari paper, scaled down only in the training budget, not the network itself.

In [ ]:
class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 512)
        self.fc2 = nn.Linear(512, n_actions)

    def forward(self, x):
        x = x / 255.0  # uint8 pixels -> 0..1
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)  # one Q-value per action, no activation - these are raw expected-reward estimates

policy_net = DQN(n_actions).to(device)
target_net = DQN(n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()
print(policy_net)
print("\nTotal parameters:", sum(p.numel() for p in policy_net.parameters()))

**Why two identical networks?** `policy_net` is the one actually being trained. `target_net` is a slow-moving copy, only updated every so often, used purely to compute *targets* for the loss. Without this split, the network would be chasing a target that moves every single step (because the target depends on the network's own output) - training against a copy that only updates occasionally makes learning far more stable. This is the **target network** trick from the DQN paper.

## Step 4 - Remembering experience: the replay buffer

> 🏷️ **B1-K1-W3** · Realiseert (onderdelen van) software — kwalificatiedossier Software development, kerntaak B1-K1

If we trained on transitions in the exact order they happen, consecutive frames are extremely similar to each other, and the network would overfit to whatever the agent is doing *right now* instead of learning something general. The fix: store the last N transitions (state, action, reward, next state, done) in a buffer, and train on **random batches** sampled from it. This breaks the correlation between consecutive training examples - the same reason `shuffle=True` mattered in the earlier DataLoader-based notebooks, just more important here because RL data is naturally much more correlated than a shuffled dataset of photos.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(a, dtype=torch.int64, device=device),
            torch.tensor(r, dtype=torch.float32, device=device),
            torch.tensor(np.array(s2), dtype=torch.float32, device=device),
            torch.tensor(d, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)

## Step 5 - Choosing actions: epsilon-greedy exploration

> 🏷️ **B1-K1-W3** · Realiseert (onderdelen van) software — kwalificatiedossier Software development, kerntaak B1-K1

Straight from the exploration/exploitation trade-off in your course book: if the agent only ever takes the action it currently *thinks* is best, it can get stuck never discovering a better strategy. **Epsilon-greedy** fixes this cheaply: with probability `epsilon`, take a completely random action instead of the network's best guess. We start with `epsilon = 1.0` (fully random - the network knows nothing yet anyway) and linearly decay it down to a small floor as training progresses, so the agent explores a lot early and mostly exploits what it's learned later.

In [ ]:
def epsilon_by_step(step, eps_start, eps_end, decay_steps):
    fraction = min(1.0, step / decay_steps)
    return eps_start + fraction * (eps_end - eps_start)

def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    with torch.no_grad():
        state_t = torch.tensor(np.array(state)[None], dtype=torch.float32, device=device)
        return policy_net(state_t).argmax(dim=1).item()

# sanity check: epsilon should start high and decay to the floor
for s in [0, 25_000, 50_000, 100_000, 200_000]:
    print(f"step {s:>7}  ->  epsilon {epsilon_by_step(s, 1.0, 0.05, 100_000):.3f}")

## Step 6 - The training loop

> 🏷️ **B1-K1-W3** · Realiseert (onderdelen van) software — kwalificatiedossier Software development, kerntaak B1-K1

This is Q-learning's core update rule, straight from the Bellman equation in your course book:

$$Q(s, a) \leftarrow r + \gamma \max_{a'} Q(s', a')$$

In words: the value of taking action `a` in state `s` should equal the reward you got, plus the discounted (`gamma`) value of the *best* action available from wherever you land next. We never know the true Q-values, so instead we treat the right-hand side (computed using the stable `target_net`) as a target, and train `policy_net` to predict it - exactly like training against a label, except the "label" is itself an estimate that gets better as training progresses.

**Before running the full budget below**, this cell first times a short burst of real environment steps on *this* runtime, so the time estimate is measured, not guessed - Colab GPU throughput varies enough between sessions that a fixed promised number would likely be wrong.

In [ ]:
# --- hyperparameters (change these and re-run to experiment - that's part of the assignment) ---
REPLAY_CAPACITY = 30_000
BATCH_SIZE = 32
GAMMA = 0.99
LR = 1e-4
EPS_START, EPS_END, EPS_DECAY_STEPS = 1.0, 0.05, 100_000
LEARNING_STARTS = 10_000       # steps of pure random play before any training starts
TRAIN_FREQ = 4                 # one gradient step every 4 environment steps
TARGET_SYNC_EVERY = 1_000      # sync target_net every this many gradient steps
TOTAL_STEPS = 500_000          # start smaller (e.g. 100_000) for your first real run
CHECKPOINT_EVERY_EPISODES = 20
CHECKPOINT_PATH = "pong_dqn_checkpoint.pt"  # point this at a Google Drive path to survive a Colab disconnect

optimizer = torch.optim.Adam(policy_net.parameters(), lr=LR)
buffer = ReplayBuffer(REPLAY_CAPACITY)

# --- measure real throughput on this runtime before committing to a long run ---
probe_env = make_env()
s, _ = probe_env.reset()
t0 = time.time()
for _ in range(500):
    s, r, term, trunc, info = probe_env.step(probe_env.action_space.sample())
    if term or trunc:
        s, _ = probe_env.reset()
elapsed = time.time() - t0
steps_per_sec = 500 / elapsed
print(f"Measured ~{steps_per_sec:.0f} environment steps/sec on this runtime.")
print(f"At that rate, {TOTAL_STEPS:,} steps would take roughly {TOTAL_STEPS/steps_per_sec/60:.0f} minutes "
      f"(the actual run will be somewhat slower once training - not just stepping - kicks in).")
probe_env.close()

In [ ]:
# Optional but recommended for a multi-hour run: mount Drive so a checkpoint
# survives a Colab disconnect. Skip this cell to just use local Colab storage
# (fine for a shorter run, but you lose progress if the runtime resets).
#
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_PATH = "/content/drive/MyDrive/pong_dqn_checkpoint.pt"

In [ ]:
import os

start_step = 0
episode_rewards = []

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    policy_net.load_state_dict(ckpt["policy_net"])
    target_net.load_state_dict(ckpt["target_net"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"]
    episode_rewards = ckpt["episode_rewards"]
    print(f"Resumed from checkpoint at step {start_step:,} ({len(episode_rewards)} episodes so far).")
else:
    print("No checkpoint found - starting fresh.")

state, info = env.reset()
current_episode_reward = 0.0
train_step_count = 0
t0 = time.time()

for step in range(start_step, TOTAL_STEPS):
    epsilon = epsilon_by_step(step, EPS_START, EPS_END, EPS_DECAY_STEPS)
    action = select_action(state, epsilon)
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    buffer.push(state, action, reward, next_state, done)
    state = next_state
    current_episode_reward += reward

    if done:
        episode_rewards.append(current_episode_reward)
        state, info = env.reset()
        current_episode_reward = 0.0

        if len(episode_rewards) % CHECKPOINT_EVERY_EPISODES == 0:
            torch.save({
                "policy_net": policy_net.state_dict(),
                "target_net": target_net.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step,
                "episode_rewards": episode_rewards,
            }, CHECKPOINT_PATH)
            recent = np.mean(episode_rewards[-20:])
            elapsed_min = (time.time() - t0) / 60
            print(f"step {step:>7,}  |  episode {len(episode_rewards):>4}  |  "
                  f"avg reward (last 20) {recent:6.2f}  |  epsilon {epsilon:.3f}  |  "
                  f"{elapsed_min:.1f} min elapsed")

    if step >= LEARNING_STARTS and step % TRAIN_FREQ == 0 and len(buffer) >= BATCH_SIZE:
        s, a, r, s2, d = buffer.sample(BATCH_SIZE)
        with torch.no_grad():
            next_q = target_net(s2).max(dim=1).values
            target = r + GAMMA * next_q * (1 - d)
        q = policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        loss = F.smooth_l1_loss(q, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_step_count += 1

        if train_step_count % TARGET_SYNC_EVERY == 0:
            target_net.load_state_dict(policy_net.state_dict())

print(f"\nDone. {len(episode_rewards)} episodes played, {train_step_count:,} gradient steps taken.")

## Step 7 - Did it learn?

> 🏷️ **B1-K1-W4** · Test software — kwalificatiedossier Software development, kerntaak B1-K1

Pong's reward per episode ranges from **-21** (you lose every point) to **+21** (you win every point). A completely random policy averages close to -21 (or the built-in game AI wins almost every rally). Progress looks like the average climbing from around -21 up toward 0 and eventually positive - not a straight line, and not fast. Be honest in your write-up about exactly where your run landed: **DeepMind's original 2015 paper trained for ~200 million frames** to reach strong play; a few hundred thousand steps on a single Colab GPU is a small fraction of that, and showing *clear upward trend* is a legitimate, defensible result even without a fully solved agent - that gap is itself worth discussing in the assignment.

In [ ]:
def moving_average(x, window=20):
    if len(x) < window:
        return np.array(x)
    return np.convolve(x, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, alpha=0.3, label="per episode")
plt.plot(range(len(episode_rewards) - len(moving_average(episode_rewards)), len(episode_rewards)),
          moving_average(episode_rewards), label="moving average (20 episodes)", linewidth=2)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("episode")
plt.ylabel("total reward")
plt.title("Episode reward over training (-21 = always lose, +21 = always win)")
plt.legend()
plt.tight_layout()
plt.show()

## Step 8 - Watch your agent play

> 🏷️ **B1-K1-W4** · Test software — kwalificatiedossier Software development, kerntaak B1-K1

Numbers on a chart are one thing - actually watching the agent play is a much more direct sanity check. This plays one full episode with the trained policy (a small amount of epsilon left in, so it doesn't get stuck repeating one mistake) and saves it as a gif you can view right here in the notebook.

In [ ]:
from PIL import Image
from IPython.display import Image as IPImage, display

watch_env = make_env(render_mode="rgb_array")
state, info = watch_env.reset()
frames = [watch_env.render()]
done = False
total_reward = 0.0

while not done:
    action = select_action(state, epsilon=0.02)
    state, reward, terminated, truncated, info = watch_env.step(action)
    total_reward += reward
    frames.append(watch_env.render())
    done = terminated or truncated

print(f"Episode reward: {total_reward}")

gif_path = "pong_agent.gif"
imgs = [Image.fromarray(f) for f in frames[::2]]  # every 2nd frame keeps the gif a reasonable size
imgs[0].save(gif_path, save_all=True, append_images=imgs[1:], duration=33, loop=0)
display(IPImage(filename=gif_path))
watch_env.close()

## Wrap-up: from notebook to written assignment

You've now built every piece the DLMAIRIL01 Pong assignment asks for, end to end: a CNN that turns pixels into features, a DQN that learns Q-values instead of a fixed label, a replay buffer and target network that make training stable, and epsilon-greedy exploration straight from the theory in your course book. Everything here connects directly to material you'll want to cite in the paper - the Bellman equation (Step 6), exploration vs. exploitation (Step 5), and function approximation replacing a Q-table (Step 3).

**For the write-up itself, be explicit about:**
- **Architecture and hyperparameters** - the exact values in the hyperparameters cell, and *why* those (e.g. why a target network, why replay, why this epsilon schedule).
- **Your actual results** - the reward curve you got, not an idealized one. If it's still trending up when training stopped, say so.
- **Honest limitations** - training budget vs. the original paper's 200M frames, what a longer run or tuned hyperparameters would likely change, and any instability you observed.
- **What you'd try next** - e.g. Double DQN, Dueling DQN, or prioritized replay, as concrete, named next steps rather than "more training."

### Kwalificatie-koppeling
Deze notebook dekt B1-K1-W2 (technisch ontwerp: de input-representatie en het gebruik van een target-netwerk), B1-K1-W3 (realiseert het netwerk, de replay buffer en de trainingslus) en B1-K1-W4 (test software: de reward-curve en een echte rollout beoordelen) uit kerntaak B1-K1 van het kwalificatiedossier mbo Software development (Crebo 23399, gewijzigd 2024). Volledige dekking over alle lessen heen: https://projectenplaats.nl/kwalificatiedossiers/software-development